# DriftSense Phase 1 model development

## tl;dr

- The session file has **665 sessions from 19 participants** and **634 usable binary labels** (192 drift, 442 aligned).
- The selected full-session diagnostic was `activity_only`. Its chronological holdout ROC-AUC was **0.587** and F1 at the development-selected threshold was **0.394**.
- No deployable early model was frozen because the checkpoint CSV was not supplied. Final-session totals are diagnostic only and are never treated as 3/5/10-minute features.


## Context & Methods

This notebook follows the study plan: uncertain labels are excluded, preprocessing is fitted inside participant-grouped folds, days 1–7 form development data, later participant-relative days form the known-participant chronological holdout, and final-session fields are blocked from early models.

### Key Assumptions

- `start_time` is UTC and participant day 1 begins on each participant's first observed calendar date.
- The supplied file's provenance (synthetic, pilot, or consented Phase 1) must be confirmed outside this file before paper claims or Phase 2 deployment.
- A 35% development prompt-rate ceiling is an explicit operational modeling choice and should be frozen in the protocol before Phase 2.


## Data

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display
from ml.model_development import run_analysis

SESSIONS = Path('D:\\1.msc\\DriftSense\\driftsense_merged.csv')
ARTIFACTS = Path('D:\\1.msc\\DriftSense\\ml\\artifacts\\phase1_model_development')

summary = run_analysis(
    sessions_path=SESSIONS,
    output_directory=ARTIFACTS,
    development_days=7,
    max_prompt_rate=0.35,
    include_random_forest=True,
    random_state=2026,
)
quality = pd.DataFrame([summary["data_quality"]]).drop(columns=["warnings", "sessions_per_participant", "cutoff_coverage_binary_labeled"])
display(quality)
display(pd.read_csv(ARTIFACTS / "cutoff_summary.csv"))


,source,sha256,rows,columns,participants,usable_binary_labels,excluded_uncertain_or_missing_labels,label_counts,drift_prevalence,duplicate_session_ids,overlapping_sessions,time_accounting_exact_rows,time_accounting_max_abs_delta_seconds,date_min,date_max,early_feature_mode,checkpoint_rows,activity_window_rows
0,D:\1.msc\DriftSense\driftsense_merged.csv,6f5fb5dcfe66a6054952daa4d10384e9019f8436aea616...,665,18,19,634,31,"{'aligned_0': 442, 'moved_away_1': 192}",0.302839,0,0,665,0.0,2026-07-14T09:11:17.761000+00:00,2026-07-30T22:07:53.093000+00:00,context_only_duration_eligibility,0,0


,cutoff_seconds,feature_mode,observable_sessions,usable_binary_labeled_sessions,aligned_0,moved_away_1,drift_prevalence,development_rows,chronological_holdout_rows,chronological_holdout_participants,common_10_minute_subset_rows
0,180,context_only_duration_eligibility,647,616,430,186,0.301948,425,191,18,528
1,300,context_only_duration_eligibility,625,595,414,181,0.304202,411,184,18,528
2,600,context_only_duration_eligibility,555,528,362,166,0.314394,362,166,18,528


## Results

In [2]:
comparison = pd.read_csv(ARTIFACTS / "model_comparison.csv")
development = comparison[
    (comparison["evaluation"] == "participant_grouped_development_oof")
    & (comparison["cohort"] == "cutoff_specific")
]
display(
    development[
        ["cutoff_seconds", "model", "n", "prevalence", "roc_auc", "brier", "f1", "prompt_rate"]
    ].sort_values(["cutoff_seconds", "roc_auc"], ascending=[True, False])
)

full_holdout = comparison[
    (comparison["cutoff_seconds"].astype(str) == "full_session_diagnostic")
    & (comparison["evaluation"] == "chronological_known_participant_holdout")
]
display(
    full_holdout[
        ["model", "n", "roc_auc", "brier", "accuracy", "precision", "recall", "f1", "prompt_rate"]
    ].sort_values("roc_auc", ascending=False)
)
display(pd.DataFrame([summary["selection"]]))


,cutoff_seconds,model,n,prevalence,roc_auc,brier,f1,prompt_rate
6,180,intended_duration,425,0.322353,0.522823,0.218602,0.000000,0.000000
3,180,fixed_timer_prompt_all,425,0.322353,0.500000,0.677647,0.487544,1.000000
9,180,task_site_domain,425,0.322353,0.498834,0.229154,0.152941,0.077647
12,180,task_type,425,0.322353,0.494640,0.224300,0.000000,0.000000
0,180,majority_class,425,0.322353,0.480599,0.218571,0.000000,0.000000
21,300,intended_duration,411,0.321168,0.516034,0.220105,0.000000,0.000000
18,300,fixed_timer_prompt_all,411,0.321168,0.500000,0.678832,0.486188,1.000000
15,300,majority_class,411,0.321168,0.477490,0.218187,0.000000,0.000000
24,300,task_site_domain,411,0.321168,0.473254,0.233048,0.084848,0.080292
27,300,task_type,411,0.321168,0.464280,0.227265,0.000000,0.009732


,model,n,roc_auc,brier,accuracy,precision,recall,f1,prompt_rate
63,task_context_activity_participant_relative,193,0.645678,0.199097,0.694301,0.409091,0.352941,0.378947,0.227979
60,task_context_activity,193,0.615300,0.194232,0.740933,0.517241,0.294118,0.375000,0.150259
52,task_site_domain,193,0.608879,0.191675,0.735751,0.500000,0.078431,0.135593,0.041451
54,task_type,193,0.592102,0.195954,0.735751,0.000000,0.000000,0.000000,0.000000
56,activity_only,193,0.586716,0.196579,0.751295,0.615385,0.156863,0.250000,0.067358
58,task_type_activity,193,0.582022,0.197622,0.735751,0.500000,0.196078,0.281690,0.103627
65,random_forest_context_activity,193,0.567661,0.228075,0.632124,0.343750,0.431373,0.382609,0.331606
50,intended_duration,193,0.526305,0.199646,0.735751,0.500000,0.039216,0.072727,0.020725
48,fixed_timer_prompt_all,193,0.500000,0.735751,0.264249,0.264249,1.000000,0.418033,1.000000
46,majority_class,193,0.500000,0.197499,0.735751,0.000000,0.000000,0.000000,0.000000


,status,reason,deployable_early_model_created,diagnostic_model,diagnostic_scope,threshold_selection,chronological_holdout,chronological_holdout_participant_bootstrap_95_ci
0,blocked_for_phase2,checkpoint_csv_missing,False,activity_only,Uses final-session totals and must not be depl...,"{'threshold': 0.35915431885150784, 'developmen...","{'n': 193, 'prevalence': 0.26424870466321243, ...","{'repetitions': 1000, 'accuracy': [0.549007709..."


## Takeaways

1. The current session table is large enough for a modest population-level pilot model, but not for a high-capacity model per participant.
2. Context-only early baselines are valid because their inputs are known at session start. The full-session activity comparison is retrospective and cannot justify a 3/5/10-minute intervention.
3. Supply the checkpoint CSV to run activity-only, context-plus-activity, and participant-relative early comparisons and unlock the frozen-model artifact. Supply activity windows as well to add recent-window features.
4. Treat discrimination, calibration, coverage, and false-prompt burden together. A model whose confidence interval includes chance should not be presented as established enrichment.
